In [10]:
"""
ABC-XYZ Spare Parts Classification System
Extension for Jan 1st Week PO Data
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

class SparePartsABCXYZClassifier:
    """
    Complete ML-based ABC-XYZ classifier for spare parts
    Extends previous framework with spare parts specific features
    """
    
    def __init__(self):
        self.abc_scaler = StandardScaler()
        self.xyz_scaler = StandardScaler()
        self.abc_model = KMeans(n_clusters=3, random_state=42)
        self.xyz_model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.is_fitted = False
        
        # SBC Thresholds (Syntetos-Boylan-Croston)
        self.adi_threshold = 1.32
        self.cv2_threshold = 0.49
        
    def load_and_prepare_data(self, file_path):
        """
        Load Jan 1st week spare parts PO data
        Expected columns: Part_Number, Description, Quantity, Unit_Price, 
                         Total_Value, Date, Supplier, Lead_Time_Days, etc.
        """
        print(f"Loading data from: {file_path}")
        df = pd.read_excel(file_path)
        
        # Standardize column names (handle variations)
        df.columns = [col.strip().replace(' ', '_').lower() for col in df.columns]
        
        print(f"Loaded {len(df)} records")
        print(f"Columns: {list(df.columns)}")
        
        return df
    
    def calculate_abc_features(self, df):
        """
        Calculate ABC classification features
        Spare parts specific: criticality, lead time, obsolescence risk
        """
        features = pd.DataFrame()
        # ensure we have a normalized part_number column
        part_col = None
        for candidate in ['part_number', 'material', 'material_description', 'part_no', 'part']:
            if candidate in df.columns:
                part_col = candidate
                break
        if part_col:
            features['part_number'] = df[part_col]
        else:
            features['part_number'] = df.index

        # helper to choose column from list of possible names
        def pick_col(possible):
            for col in possible:
                if col in df.columns:
                    return col
            return None

        # Core value metrics (revenue)
        rev_col = pick_col(['total_value', 'net_value_(item)', 'net_value_(header)', 'net_price',
                            'net_value', 'value', 'order_value'])
        qty_col = pick_col(['quantity', 'order_quantity_(item)', 'order_quantity', 'qty', 'af_open_order_quantity'])
        price_col = pick_col(['unit_price', 'net_price', 'price'])

        if rev_col is not None:
            features['revenue'] = df[rev_col]
        elif qty_col is not None and price_col is not None:
            features['revenue'] = df[qty_col] * df[price_col]
        else:
            raise ValueError("Need total_value or (quantity + unit_price) columns")
        
        # Calculate derived metrics
        total_revenue = features['revenue'].sum()
        features['revenue_contribution'] = features['revenue'] / total_revenue
        
        # Spare parts specific: Annual usage if available
        if 'annual_usage' in df.columns:
            features['annual_usage'] = df['annual_usage']
            features['usage_value'] = features['annual_usage'] * df.get('unit_price', 1)
        else:
            # Estimate from current PO
            features['annual_usage'] = features['revenue'] * 52  # Weekly to annual estimate
            features['usage_value'] = features['annual_usage']
        
        # Criticality scoring (if available)
        if 'criticality' in df.columns:
            # Map criticality to numeric (A=3, B=2, C=1)
            crit_map = {'A': 3, 'B': 2, 'C': 1, 'Critical': 3, 'High': 2.5, 'Medium': 2, 'Low': 1}
            features['criticality_score'] = df['criticality'].map(crit_map).fillna(2)
        else:
            features['criticality_score'] = 2  # Default medium
        
        # Lead time impact
        if 'lead_time_days' in df.columns:
            features['lead_time'] = df['lead_time_days']
            features['lead_time_criticality'] = np.where(df['lead_time_days'] > 30, 3,
                                               np.where(df['lead_time_days'] > 14, 2, 1))
        else:
            features['lead_time'] = 14
            features['lead_time_criticality'] = 2
        
        # Stockout cost estimation (revenue * criticality)
        features['stockout_cost'] = features['revenue'] * features['criticality_score']
        
        # Composite ABC score (weighted combination)
        features['abc_score'] = (
            features['revenue_contribution'] * 0.4 +
            (features['criticality_score'] / 3) * 0.3 +
            (features['lead_time_criticality'] / 3) * 0.2 +
            (features['stockout_cost'] / features['stockout_cost'].max()) * 0.1
        )
        
        return features
    
    def calculate_xyz_features(self, df, historical_data=None):
        """
        Calculate XYZ (demand variability) features
        For spare parts: ADI, CV², and additional intermittency measures
        """
        features = pd.DataFrame()
        # find part_number column consistently
        part_col = None
        for candidate in ['part_number', 'material', 'material_description', 'part_no', 'part']:
            if candidate in df.columns:
                part_col = candidate
                break
        if part_col:
            features['part_number'] = df[part_col]
        else:
            features['part_number'] = df.index
        
        # If we have historical demand data
        if historical_data is not None:
            # ensure historical_data also has standard part_number field
            if 'part_number' not in historical_data.columns:
                # try to rename similar column
                for c in ['material','part_no','part']:
                    if c in historical_data.columns:
                        historical_data = historical_data.rename(columns={c:'part_number'})
                        break
            xyz_features = self._calculate_historical_variability(historical_data)
            features = features.merge(xyz_features, on='part_number', how='left')
        else:
            # Estimate from current data patterns
            features = self._estimate_variability_from_po(df, features)
        
        # Additional spare parts specific features
        if 'quantity' in df.columns:
            qty = df['quantity']
            features['order_quantity'] = qty
            features['quantity_cv'] = qty.std() / qty.mean() if qty.mean() > 0 else 0
        
        # Supplier reliability (if available)
        if 'supplier' in df.columns:
            supplier_counts = df['supplier'].value_counts()
            features['supplier_diversity'] = df.groupby('part_number')['supplier'].nunique().values
            
        return features
    
    def _calculate_historical_variability(self, historical_df):
        """
        Calculate ADI and CV² from historical demand data
        historical_df should have: part_number, date, quantity
        """
        results = []
        
        for part in historical_df['part_number'].unique():
            part_data = historical_df[historical_df['part_number'] == part].sort_values('date')
            demand_series = part_data['quantity'].values
            
            # ADI: Average Inter-Demand Interval
            non_zero_indices = np.where(demand_series > 0)[0]
            if len(non_zero_indices) > 1:
                intervals = np.diff(non_zero_indices)
                adi = np.mean(intervals)
            else:
                adi = len(demand_series)  # No demand or single demand
            
            # CV²: Squared Coefficient of Variation
            non_zero_demand = demand_series[demand_series > 0]
            if len(non_zero_demand) > 1:
                cv = np.std(non_zero_demand) / np.mean(non_zero_demand)
                cv2 = cv ** 2
            else:
                cv2 = 0
            
            # Additional metrics
            zero_demand_ratio = np.sum(demand_series == 0) / len(demand_series)
            
            results.append({
                'part_number': part,
                'adi': adi,
                'cv2': cv2,
                'zero_demand_ratio': zero_demand_ratio,
                'demand_count': len(non_zero_demand),
                'total_periods': len(demand_series)
            })
        
        return pd.DataFrame(results)
    
    def _estimate_variability_from_po(self, df, features):
        """
        Estimate variability when historical data isn't available
        Use PO characteristics as proxies
        """
        # Default moderate variability
        n_parts = len(features)
        features['adi'] = np.random.uniform(1.0, 2.0, n_parts)  # Placeholder
        features['cv2'] = np.random.uniform(0.3, 0.6, n_parts)  # Placeholder
        
        # Better estimates if we have multiple POs for same part
        # locate part_number column
        part_col = 'part_number' if 'part_number' in df.columns else None
        if part_col is None:
            for candidate in ['material','material_description','part_no','part']:
                if candidate in df.columns:
                    part_col = candidate
                    break
        # locate quantity column
        qty_col = None
        for candidate in ['quantity', 'order_quantity_(item)', 'order_quantity', 'qty', 'af_open_order_quantity']:
            if candidate in df.columns:
                qty_col = candidate
                break
        if part_col is not None and qty_col is not None:
            part_stats = df.groupby(part_col).agg({
                qty_col: ['count', 'std', 'mean']
            }).reset_index()
            # rename to standardized names
            part_stats.columns = [part_col, 'po_count', 'qty_std', 'qty_mean']
            part_stats['qty_cv'] = part_stats['qty_std'] / part_stats['qty_mean']
            # merge using part_col, then rename to unified field
            part_stats = part_stats.rename(columns={part_col: 'part_number'})
            features = features.merge(part_stats, on='part_number', how='left')
            
            # Estimate ADI from PO frequency
            features['adi'] = 52 / features['po_count'].fillna(1)  # Weeks between orders
            features['cv2'] = (features['qty_cv'].fillna(0.5)) ** 2
        
        features['zero_demand_ratio'] = 0.3  # Default estimate
        features['demand_count'] = 1
        features['total_periods'] = 52
        
        return features
    
    def classify_abc_ml(self, abc_features):
        """
        ML-based ABC classification using K-Means
        """
        # Prepare features for clustering
        feature_cols = ['revenue_contribution', 'criticality_score', 
                       'lead_time_criticality', 'stockout_cost']
        
        X = abc_features[feature_cols].fillna(0)
        X_scaled = self.abc_scaler.fit_transform(X)
        
        # Fit K-Means
        self.abc_model.fit(X_scaled)
        labels = self.abc_model.labels_
        
        # Determine which cluster is A, B, C based on revenue
        cluster_revenue = abc_features.groupby(labels)['revenue'].mean().sort_values(ascending=False)
        cluster_map = {cluster_revenue.index[0]: 'A',
                      cluster_revenue.index[1]: 'B',
                      cluster_revenue.index[2]: 'C'}
        
        abc_features['abc_class'] = [cluster_map[l] for l in labels]
        abc_features['abc_cluster'] = labels
        
        # Calculate confidence (distance to cluster center)
        distances = self.abc_model.transform(X_scaled)
        min_dist = np.min(distances, axis=1)
        abc_features['abc_confidence'] = 1 - (min_dist / np.max(min_dist))
        
        return abc_features
    
    def classify_abc_traditional(self, abc_features):
        """
        Traditional Pareto-based ABC classification
        """
        # Sort by revenue contribution
        abc_features = abc_features.sort_values('revenue_contribution', ascending=False)
        abc_features['cumulative_revenue'] = abc_features['revenue_contribution'].cumsum()
        
        # Apply Pareto rules
        conditions = [
            abc_features['cumulative_revenue'] <= 0.8,
            (abc_features['cumulative_revenue'] > 0.8) & 
            (abc_features['cumulative_revenue'] <= 0.95),
            abc_features['cumulative_revenue'] > 0.95
        ]
        choices = ['A', 'B', 'C']
        
        abc_features['abc_class'] = np.select(conditions, choices, default='C')
        
        return abc_features
    
    def classify_xyz_sbc(self, xyz_features):
        """
        Syntetos-Boylan-Croston (SBC) classification
        Industry standard for demand pattern classification
        """
        def sbc_classifier(row):
            adi = row['adi']
            cv2 = row['cv2']
            
            if adi < self.adi_threshold and cv2 < self.cv2_threshold:
                return 'Smooth'
            elif adi >= self.adi_threshold and cv2 < self.cv2_threshold:
                return 'Intermittent'
            elif adi < self.adi_threshold and cv2 >= self.cv2_threshold:
                return 'Erratic'
            else:
                return 'Lumpy'
        
        xyz_features['xyz_class'] = xyz_features.apply(sbc_classifier, axis=1)
        
        # Add pattern characteristics
        pattern_desc = {
            'Smooth': 'Regular, predictable demand',
            'Intermittent': 'Sporadic demand with low variability',
            'Erratic': 'Frequent demand with high variability',
            'Lumpy': 'Sporadic demand with high variability'
        }
        xyz_features['pattern_description'] = xyz_features['xyz_class'].map(pattern_desc)
        
        return xyz_features
    
    def classify_xyz_ml(self, xyz_features, training_data=None):
        """
        ML-enhanced XYZ classification
        Uses Random Forest with additional features beyond ADI/CV2
        """
        if training_data is not None:
            # Train model
            X = training_data[['adi', 'cv2', 'zero_demand_ratio', 'demand_count']]
            y = training_data['xyz_class']
            
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
            X_train_scaled = self.xyz_scaler.fit_transform(X_train)
            self.xyz_model.fit(X_train_scaled, y_train)
            self.is_fitted = True
        
        if self.is_fitted:
            # Predict
            X_new = xyz_features[['adi', 'cv2', 'zero_demand_ratio', 'demand_count']].fillna(0)
            X_new_scaled = self.xyz_scaler.transform(X_new)
            
            xyz_features['xyz_class'] = self.xyz_model.predict(X_new_scaled)
            probs = self.xyz_model.predict_proba(X_new_scaled)
            xyz_features['xyz_confidence'] = np.max(probs, axis=1)
        else:
            # Fall back to SBC rules
            xyz_features = self.classify_xyz_sbc(xyz_features)
            xyz_features['xyz_confidence'] = 0.8  # Default confidence
        
        return xyz_features
    
    def create_matrix(self, abc_features, xyz_features):
        """
        Create ABC-XYZ classification matrix
        """
        # Merge classifications
        matrix = abc_features.merge(xyz_features, on='part_number', how='outer')
        
        # Create ABC-XYZ code
        matrix['abc_xyz'] = matrix['abc_class'] + '-' + matrix['xyz_class'].str[0]
        
        # Assign inventory policies
        policies = {
            'A-S': {'policy': 'Continuous Review (R,Q)', 'safety_stock_days': 7, 
                   'review_period': 'Daily', 'forecast_method': 'ARIMA/ETS'},
            'A-I': {'policy': 'Continuous Review + Croston', 'safety_stock_days': 14,
                   'review_period': 'Daily', 'forecast_method': 'Croston/SBA'},
            'A-E': {'policy': 'Dynamic Safety Stock', 'safety_stock_days': 21,
                   'review_period': 'Daily', 'forecast_method': 'ML Ensemble'},
            'A-L': {'policy': 'High Safety Stock + VMI', 'safety_stock_days': 30,
                   'review_period': 'Daily', 'forecast_method': 'Bootstrap'},
            'B-S': {'policy': 'Periodic Review (T,S)', 'safety_stock_days': 14,
                   'review_period': 'Weekly', 'forecast_method': 'Exponential Smoothing'},
            'B-I': {'policy': 'Periodic Review + SBA', 'safety_stock_days': 21,
                   'review_period': 'Weekly', 'forecast_method': 'SBA'},
            'B-E': {'policy': 'Adaptive Review', 'safety_stock_days': 30,
                   'review_period': 'Weekly', 'forecast_method': 'Random Forest'},
            'B-L': {'policy': 'Two-Bin System', 'safety_stock_days': 45,
                   'review_period': 'Bi-weekly', 'forecast_method': 'Heuristic'},
            'C-S': {'policy': 'Min-Max System', 'safety_stock_days': 30,
                   'review_period': 'Monthly', 'forecast_method': 'Moving Average'},
            'C-I': {'policy': 'Order-up-to Level', 'safety_stock_days': 45,
                   'review_period': 'Monthly', 'forecast_method': 'Naive'},
            'C-E': {'policy': 'Make-to-Order', 'safety_stock_days': 60,
                   'review_period': 'Monthly', 'forecast_method': 'None'},
            'C-L': {'policy': 'Drop-ship/Discontinue', 'safety_stock_days': 90,
                   'review_period': 'Quarterly', 'forecast_method': 'None'}
        }
        
        # Apply policies
        matrix['inventory_policy'] = matrix['abc_xyz'].map(
            lambda x: policies.get(x, policies.get('C-L'))['policy']
        )
        matrix['safety_stock_days'] = matrix['abc_xyz'].map(
            lambda x: policies.get(x, policies.get('C-L'))['safety_stock_days']
        )
        matrix['review_period'] = matrix['abc_xyz'].map(
            lambda x: policies.get(x, policies.get('C-L'))['review_period']
        )
        matrix['forecast_method'] = matrix['abc_xyz'].map(
            lambda x: policies.get(x, policies.get('C-L'))['forecast_method']
        )
        
        # Calculate recommended safety stock
        if 'lead_time' in matrix.columns:
            matrix['safety_stock_quantity'] = (
                matrix['safety_stock_days'] / 7 * 
                matrix.get('order_quantity', 1) / 
                matrix.get('po_count', 1)
            ).fillna(0)
        
        return matrix
    
    def generate_report(self, matrix):
        """
        Generate comprehensive classification report
        """
        report = {
            'summary_stats': {
                'total_parts': len(matrix),
                'total_value': matrix['revenue'].sum(),
                'avg_lead_time': matrix['lead_time'].mean()
            },
            'abc_distribution': matrix['abc_class'].value_counts().to_dict(),
            'xyz_distribution': matrix['xyz_class'].value_counts().to_dict(),
            'matrix_distribution': matrix['abc_xyz'].value_counts().to_dict(),
            'high_priority': matrix[matrix['abc_class'] == 'A']['part_number'].tolist(),
            'critical_attention': matrix[
                (matrix['abc_class'] == 'A') & 
                (matrix['xyz_class'].isin(['Erratic', 'Lumpy']))
            ]['part_number'].tolist()
        }
        
        return report
    
    def run_full_classification(self, file_path, historical_data=None, use_ml_xyz=False):
        """
        Execute complete classification pipeline
        """
        print("=" * 60)
        print("SPARE PARTS ABC-XYZ CLASSIFICATION SYSTEM")
        print("=" * 60)
        
        # 1. Load data
        df = self.load_and_prepare_data(file_path)
        
        # 2. Calculate ABC features
        print("\n[1/4] Calculating ABC features...")
        abc_features = self.calculate_abc_features(df)
        
        # 3. Calculate XYZ features
        print("[2/4] Calculating XYZ features...")
        xyz_features = self.calculate_xyz_features(df, historical_data)
        
        # 4. Classify ABC
        print("[3/4] Classifying ABC...")
        abc_features = self.classify_abc_ml(abc_features)
        
        # 5. Classify XYZ
        print("[4/4] Classifying XYZ...")
        if use_ml_xyz and historical_data is not None:
            xyz_features = self.classify_xyz_ml(xyz_features, historical_data)
        else:
            xyz_features = self.classify_xyz_sbc(xyz_features)
        
        # 6. Create matrix
        print("\n[5/4] Creating ABC-XYZ Matrix...")
        matrix = self.create_matrix(abc_features, xyz_features)
        
        # 7. Generate report
        report = self.generate_report(matrix)
        
        # Display results
        print("\n" + "=" * 60)
        print("CLASSIFICATION RESULTS")
        print("=" * 60)
        
        print(f"\n📊 ABC Distribution:")
        for cls, count in report['abc_distribution'].items():
            pct = count / report['summary_stats']['total_parts'] * 100
            print(f"   {cls}: {count} parts ({pct:.1f}%)")
        
        print(f"\n📈 XYZ Distribution:")
        for cls, count in report['xyz_distribution'].items():
            pct = count / report['summary_stats']['total_parts'] * 100
            print(f"   {cls}: {count} parts ({pct:.1f}%)")
        
        print(f"\n🎯 ABC-XYZ Matrix:")
        for cls, count in sorted(report['matrix_distribution'].items()):
            pct = count / report['summary_stats']['total_parts'] * 100
            print(f"   {cls}: {count} parts ({pct:.1f}%)")
        
        print(f"\n⚠️  Critical Items (A-class + Erratic/Lumpy): {len(report['critical_attention'])}")
        if len(report['critical_attention']) > 0:
            print(f"   Part numbers: {', '.join(map(str, report['critical_attention'][:5]))}")
            if len(report['critical_attention']) > 5:
                print(f"   ... and {len(report['critical_attention']) - 5} more")
        
        return matrix, report


# ============================================
# EXECUTION SCRIPT
# ============================================

if __name__ == "__main__":
    # Initialize classifier
    classifier = SparePartsABCXYZClassifier()
    
    # Run classification
    # Replace with your actual file path
    file_path = "Jan 1st week Spare Parts PO.xlsx"
    
    try:
        matrix, report = classifier.run_full_classification(
            file_path=file_path,
            historical_data=None,  # Add if you have historical demand data
            use_ml_xyz=False       # Set True if you have training data
        )
        
        # Save results
        output_file = "ABC_XYZ_Classification_Results.xlsx"
        matrix.to_excel(output_file, index=False)
        print(f"\n✅ Results saved to: {output_file}")
        
        # Display sample of classified items
        print("\n" + "=" * 60)
        print("SAMPLE CLASSIFIED ITEMS")
        print("=" * 60)
        sample_cols = ['part_number', 'abc_class', 'xyz_class', 'abc_xyz', 
                      'inventory_policy', 'safety_stock_days', 'revenue']
        print(matrix[sample_cols].head(10).to_string())
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("\nPlease ensure your Excel file has the required columns:")
        print("Required: part_number (or similar), quantity, unit_price/total_value")
        print("Optional: lead_time_days, criticality, supplier, annual_usage")

SPARE PARTS ABC-XYZ CLASSIFICATION SYSTEM
Loading data from: Jan 1st week Spare Parts PO.xlsx
Loaded 1629 records
Columns: ['customer_reference', 'document_date', 'sales_document_type', 'sales_document', 'sales_document_item', 'sold-to_party', 'material', 'order_quantity_(item)', 'sales_unit', 'net_value_(item)', 'document_currency', 'order_reason', 'customer_reference_(header)', 'created_by', 'created_on', 'time', 'billing_block', 'sales_document_description', 'exchange_rate_type', 'delivery_block', 'net_value_(header)', 'division', 'sd_document_category', 'sales_office', 'sales_group', 'sales_organization', 'distribution_channel', 'overall_status', 'overall_delivery_status_(all_items)', 'sold-to_party_name', 'reason_for_rejection', 'material_description', 'batch', 'confirmed_quantity_(item)', 'pricing_unit', 'pricing_unit_(per)', 'storage_location', 'base_unit_of_measure', 'net_price', 'returns', 'shipping_point/receiving_pt', 'plant', 'target_quantity', 'overall_status_item', 'overa

  File "d:\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "d:\anaconda\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "d:\anaconda\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\anaconda\Lib\subprocess.py", line 1554, in _execute_child
    hp, ht, pid, t


[5/4] Creating ABC-XYZ Matrix...

CLASSIFICATION RESULTS

📊 ABC Distribution:
   C: 8320 parts (89.1%)
   B: 746 parts (8.0%)
   A: 275 parts (2.9%)

📈 XYZ Distribution:
   Intermittent: 5664 parts (60.6%)
   Lumpy: 3677 parts (39.4%)

🎯 ABC-XYZ Matrix:
   A-I: 106 parts (1.1%)
   A-L: 169 parts (1.8%)
   B-I: 321 parts (3.4%)
   B-L: 425 parts (4.5%)
   C-I: 5237 parts (56.1%)
   C-L: 3083 parts (33.0%)

⚠️  Critical Items (A-class + Erratic/Lumpy): 169
   Part numbers: 1GC-E7641-10-00, 1GC-E7641-10-00, 1GC-E7641-10-00, 1GC-E7641-10-00, 1GC-E7641-10-00
   ... and 164 more

✅ Results saved to: ABC_XYZ_Classification_Results.xlsx

SAMPLE CLASSIFIED ITEMS
       part_number abc_class     xyz_class abc_xyz       inventory_policy  safety_stock_days   revenue
0     18S-E5570-01         B  Intermittent     B-I  Periodic Review + SBA                 21  25588.98
1     18S-E5570-01         B  Intermittent     B-I  Periodic Review + SBA                 21  25588.98
2     18S-E5570-01         B

In [11]:
# Basic usage
classifier = SparePartsABCXYZClassifier()
matrix, report = classifier.run_full_classification("Jan 1st week Spare Parts PO.xlsx")

SPARE PARTS ABC-XYZ CLASSIFICATION SYSTEM
Loading data from: Jan 1st week Spare Parts PO.xlsx
Loaded 1629 records
Columns: ['customer_reference', 'document_date', 'sales_document_type', 'sales_document', 'sales_document_item', 'sold-to_party', 'material', 'order_quantity_(item)', 'sales_unit', 'net_value_(item)', 'document_currency', 'order_reason', 'customer_reference_(header)', 'created_by', 'created_on', 'time', 'billing_block', 'sales_document_description', 'exchange_rate_type', 'delivery_block', 'net_value_(header)', 'division', 'sd_document_category', 'sales_office', 'sales_group', 'sales_organization', 'distribution_channel', 'overall_status', 'overall_delivery_status_(all_items)', 'sold-to_party_name', 'reason_for_rejection', 'material_description', 'batch', 'confirmed_quantity_(item)', 'pricing_unit', 'pricing_unit_(per)', 'storage_location', 'base_unit_of_measure', 'net_price', 'returns', 'shipping_point/receiving_pt', 'plant', 'target_quantity', 'overall_status_item', 'overa

In [6]:
# inspect column names directly from the spreadsheet
import pandas as pd
_df = pd.read_excel(r"Jan 1st week Spare Parts PO.xlsx")
cols = [c.strip().replace(' ','_').lower() for c in _df.columns]
print(cols)


['customer_reference', 'document_date', 'sales_document_type', 'sales_document', 'sales_document_item', 'sold-to_party', 'material', 'order_quantity_(item)', 'sales_unit', 'net_value_(item)', 'document_currency', 'order_reason', 'customer_reference_(header)', 'created_by', 'created_on', 'time', 'billing_block', 'sales_document_description', 'exchange_rate_type', 'delivery_block', 'net_value_(header)', 'division', 'sd_document_category', 'sales_office', 'sales_group', 'sales_organization', 'distribution_channel', 'overall_status', 'overall_delivery_status_(all_items)', 'sold-to_party_name', 'reason_for_rejection', 'material_description', 'batch', 'confirmed_quantity_(item)', 'pricing_unit', 'pricing_unit_(per)', 'storage_location', 'base_unit_of_measure', 'net_price', 'returns', 'shipping_point/receiving_pt', 'plant', 'target_quantity', 'overall_status_item', 'overall_delivery_status_(item)', 'exchange_rate', 'pricing_date', 'personnel_partner_function', 'personnel_number', 'partner_fun

In [7]:
# test the column-picking logic interactively
import pandas as pd
_test = pd.read_excel(r"Jan 1st week Spare Parts PO.xlsx")
cols = [c.strip().replace(' ','_').lower() for c in _test.columns]
print(cols[:20])

# define pick_col same as in class
def pick_col(possible):
    for col in possible:
        if col in cols:
            return col
    return None

print('rev_col', pick_col(['total_value', 'net_value_(item)', 'net_value_(header)', 'net_price',
                            'net_value', 'value', 'order_value']))
print('qty_col', pick_col(['quantity', 'order_quantity_(item)', 'order_quantity', 'qty', 'af_open_order_quantity']))
print('price_col', pick_col(['unit_price', 'net_price', 'price']))


['customer_reference', 'document_date', 'sales_document_type', 'sales_document', 'sales_document_item', 'sold-to_party', 'material', 'order_quantity_(item)', 'sales_unit', 'net_value_(item)', 'document_currency', 'order_reason', 'customer_reference_(header)', 'created_by', 'created_on', 'time', 'billing_block', 'sales_document_description', 'exchange_rate_type', 'delivery_block']
rev_col net_value_(item)
qty_col order_quantity_(item)
price_col net_price
